In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import random, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from datetime import timedelta
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

import wandb
wandb.login(key="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")

try:
    from tqdm.auto import tqdm as _tqdm
    def _make_bar(loader, desc):
        return _tqdm(loader, desc=desc, leave=False,
                     bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")
except Exception:
    def _make_bar(loader, desc):
        return loader

print("All imports OK")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ahmad-nafees-tihami (ahmad-nafees-tihami-brac-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


All imports OK
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


In [2]:
# Ablation Config
DATASET_NAME   = "dataset1_brisc"
TRAIN_IMG_DIR  = '/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/segmentation_task/train/images'
TRAIN_MASK_DIR = '/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/segmentation_task/train/masks'
TEST_IMG_DIR   = '/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/segmentation_task/test/images'
TEST_MASK_DIR  = '/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/segmentation_task/test/masks'

SEED       = 2024
FOLD       = 2
IMG_SIZE   = 256
BATCH_SIZE = 4
LR         = 0.0001
BASE_CH    = 64
DROPOUT    = 0.3
EPOCHS     = 50
PATIENCE   = 7

SAVE_DIR = '/kaggle/working/ablation_ckpts'
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"Seed=2024 | Fold=2 | BS=4 | LR=0.0001")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)
print("Config OK")


Device  : cuda
GPU     : Tesla T4
Seed=2024 | Fold=2 | BS=4 | LR=0.0001
Config OK


In [3]:
# ── CELL 4: Dataset ──────────────────────────────────────────
class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_size=256, augment=False):
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.img_size  = img_size
        self.augment   = augment
        exts = ('.png','.jpg','.jpeg','.tif','.tiff','.bmp')
        self.images = sorted([f for f in os.listdir(img_dir)  if f.lower().endswith(exts)])
        self.masks  = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(exts)])
        n = min(len(self.images), len(self.masks))
        self.images, self.masks = self.images[:n], self.masks[:n]

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img  = Image.open(os.path.join(self.img_dir,  self.images[idx])).convert('RGB')
        mask = Image.open(os.path.join(self.mask_dir, self.masks[idx])).convert('L')
        img  = img.resize( (self.img_size, self.img_size), Image.BILINEAR)
        mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)

        img_np  = np.array(img,  dtype=np.float32) / 255.0
        mask_np = np.array(mask, dtype=np.float32)
        if mask_np.max() > 1: mask_np /= 255.0
        mask_np = (mask_np > 0.5).astype(np.float32)

        if self.augment:
            if random.random() > 0.5: img_np = np.fliplr(img_np).copy();  mask_np = np.fliplr(mask_np).copy()
            if random.random() > 0.5: img_np = np.flipud(img_np).copy();  mask_np = np.flipud(mask_np).copy()
            if random.random() > 0.5:
                k = random.randint(1,3)
                img_np  = np.rot90(img_np,  k).copy()
                mask_np = np.rot90(mask_np, k).copy()
            if random.random() > 0.5: img_np = np.clip(img_np * random.uniform(0.8,1.2), 0, 1)
            if random.random() > 0.5:
                m = img_np.mean()
                img_np = np.clip((img_np - m) * random.uniform(0.8,1.2) + m, 0, 1)
            if random.random() > 0.7: img_np = np.clip(img_np + np.random.normal(0,0.02,img_np.shape), 0, 1)

        img_t  = torch.from_numpy(img_np).permute(2,0,1).float() * 2 - 1
        mask_t = torch.from_numpy(mask_np).unsqueeze(0).float()
        return img_t, mask_t


In [4]:
# ── CELL 5: Model ────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(nn.Conv2d(c,c//r,1,bias=False), nn.ReLU(True), nn.Conv2d(c//r,c,1,bias=False))
        self.sig = nn.Sigmoid()
    def forward(self,x): return x * self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2,1,k,padding=k//2,bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self,x):
        return x * self.sig(self.conv(torch.cat([x.mean(1,True), x.max(1,True)[0]], 1)))

class CBAM(nn.Module):
    def __init__(self,c,r=16): super().__init__(); self.ca=ChannelAttention(c,r); self.sa=SpatialAttention()
    def forward(self,x): return self.sa(self.ca(x))

class AttentionGate(nn.Module):
    def __init__(self,Fg,Fl,Fi):
        super().__init__()
        self.Wg  = nn.Sequential(nn.Conv2d(Fg,Fi,1,bias=True), nn.BatchNorm2d(Fi))
        self.Wx  = nn.Sequential(nn.Conv2d(Fl,Fi,1,bias=True), nn.BatchNorm2d(Fi))
        self.psi = nn.Sequential(nn.Conv2d(Fi,1, 1,bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu= nn.ReLU(True)

    def forward(self, g, x):
        g_up = F.interpolate(self.Wg(g), size=x.shape[2:], mode='bilinear', align_corners=False)
        return x * self.psi(self.relu(g_up + self.Wx(x)))

class ConvBlock(nn.Module):
    def __init__(self,ic,oc,dr=0.3):
        super().__init__()
        self.c1   = nn.Sequential(nn.Conv2d(ic,oc,3,padding=1,bias=False), nn.BatchNorm2d(oc), nn.ReLU(True), nn.Dropout2d(dr))
        self.c2   = nn.Sequential(nn.Conv2d(oc,oc,3,padding=1,bias=False), nn.BatchNorm2d(oc), nn.ReLU(True), nn.Dropout2d(dr))
        self.cbam = CBAM(oc)
        self.skip = nn.Sequential(nn.Conv2d(ic,oc,1,bias=False), nn.BatchNorm2d(oc)) if ic!=oc else nn.Identity()
    def forward(self,x): return self.cbam(self.c2(self.c1(x))) + self.skip(x)

class ICONNET(nn.Module):
    def __init__(self, ic=3, oc=1, base=64, dr=0.3):
        super().__init__()
        b = base
        self.init   = nn.Sequential(nn.Conv2d(ic,b,3,padding=1,bias=False), nn.BatchNorm2d(b), nn.ReLU(True))
        self.e1=ConvBlock(b,b,dr);   self.p1=nn.MaxPool2d(2)
        self.e2=ConvBlock(b,b*2,dr); self.p2=nn.MaxPool2d(2)
        self.e3=ConvBlock(b*2,b*4,dr);self.p3=nn.MaxPool2d(2)
        self.e4=ConvBlock(b*4,b*8,dr);self.p4=nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(b*8,b*16,dr)
        self.ag4=AttentionGate(b*16,b*8,b*4)
        self.ag3=AttentionGate(b*8, b*4,b*2)
        self.ag2=AttentionGate(b*4, b*2,b)
        self.ag1=AttentionGate(b*2, b,  b//2)
        self.up4=nn.ConvTranspose2d(b*16,b*8,2,stride=2); self.d4=ConvBlock(b*16,b*8,dr)
        self.up3=nn.ConvTranspose2d(b*8, b*4,2,stride=2); self.d3=ConvBlock(b*8, b*4,dr)
        self.up2=nn.ConvTranspose2d(b*4, b*2,2,stride=2); self.d2=ConvBlock(b*4, b*2,dr)
        self.up1=nn.ConvTranspose2d(b*2, b,  2,stride=2); self.d1=ConvBlock(b*2, b,  dr)
        self.out = nn.Conv2d(b, oc, 1)
        # deep supervision heads
        self.ds4 = nn.Conv2d(b*8,oc,1); self.ds3 = nn.Conv2d(b*4,oc,1); self.ds2 = nn.Conv2d(b*2,oc,1)

    def forward(self,x):
        x0 = self.init(x)
        e1 = self.e1(x0); e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2)); e4 = self.e4(self.p3(e3))
        bt = self.bottleneck(self.p4(e4))
        d4 = self.d4(torch.cat([self.up4(bt), self.ag4(bt,e4)],1))
        d3 = self.d3(torch.cat([self.up3(d4), self.ag3(d4,e3)],1))
        d2 = self.d2(torch.cat([self.up2(d3), self.ag2(d3,e2)],1))
        d1 = self.d1(torch.cat([self.up1(d2), self.ag1(d2,e1)],1))
        main = torch.sigmoid(self.out(d1))
        if self.training:
            s = x.shape[2:]
            ds4 = torch.sigmoid(F.interpolate(self.ds4(d4), s, mode='bilinear', align_corners=False))
            ds3 = torch.sigmoid(F.interpolate(self.ds3(d3), s, mode='bilinear', align_corners=False))
            ds2 = torch.sigmoid(F.interpolate(self.ds2(d2), s, mode='bilinear', align_corners=False))
            return main, ds4, ds3, ds2
        return main
        

In [5]:
# ── CELL 6: Loss ─────────────────────────────────────────────
class CombinedLoss(nn.Module):
    def __init__(self, dw=0.5, bw=0.3, fw=0.2, smooth=1e-6):
        super().__init__(); self.dw=dw; self.bw=bw; self.fw=fw; self.s=smooth

    def dice(self,p,t):
        pf,tf = p.view(-1), t.view(-1)
        return 1-(2*(pf*tf).sum()+self.s)/(pf.sum()+tf.sum()+self.s)

    def focal(self,p,t,a=0.25,g=2.0):
        bce = F.binary_cross_entropy(p,t,reduction='none')
        return (a*(1-torch.exp(-bce))**g*bce).mean()

    def forward(self,p,t):
        bce = F.binary_cross_entropy(p,t)
        return self.dw*self.dice(p,t) + self.bw*bce + self.fw*self.focal(p,t)


In [6]:
# ── CELL 7: Metrics ──────────────────────────────────────────
def compute_metrics(preds_prob, targets, threshold=0.5):
    preds = (preds_prob > threshold).astype(np.uint8)
    tflat = targets.astype(np.uint8).flatten()
    pflat = preds.flatten()
    tp = np.sum((pflat==1)&(tflat==1)); tn = np.sum((pflat==0)&(tflat==0))
    fp = np.sum((pflat==1)&(tflat==0)); fn = np.sum((pflat==0)&(tflat==1))
    smooth = 1e-6
    dice = (2*tp+smooth)/(2*tp+fp+fn+smooth)
    iou  = (tp+smooth)/(tp+fp+fn+smooth)
    acc  = (tp+tn+smooth)/(tp+tn+fp+fn+smooth)
    prec = (tp+smooth)/(tp+fp+smooth)
    rec  = (tp+smooth)/(tp+fn+smooth)
    spec = (tn+smooth)/(tn+fp+smooth)
    f1   = (2*prec*rec+smooth)/(prec+rec+smooth)
    try:    auc_roc = roc_auc_score(tflat, preds_prob.flatten())
    except: auc_roc = 0.0
    try:    auc_pr  = average_precision_score(tflat, preds_prob.flatten())
    except: auc_pr  = 0.0
    return dict(dice=dice, iou=iou, accuracy=acc, precision=prec,
                recall=rec, specificity=spec, f1=f1, auc_roc=auc_roc, auc_pr=auc_pr)

In [7]:
# CELL 8: Train / Eval Epoch — tqdm batch bar with plain-print fallback
try:
    from tqdm.auto import tqdm as _tqdm
    def _make_bar(loader, desc):
        return _tqdm(loader, desc=desc, leave=False,
                     bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")
except Exception:
    def _make_bar(loader, desc):
        return loader   # no-op fallback

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0; all_p = []; all_t = []
    for imgs, masks in _make_bar(loader, "train"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        if isinstance(out, tuple):
            main, ds4, ds3, ds2 = out
            loss = (criterion(main, masks) * 0.6 +
                    criterion(ds4,  masks) * 0.2 +
                    criterion(ds3,  masks) * 0.1 +
                    criterion(ds2,  masks) * 0.1)
        else:
            main = out
            loss = criterion(main, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_p.append(main.detach().cpu().numpy())
        all_t.append(masks.cpu().numpy())
    preds = np.concatenate(all_p)
    tgts  = np.concatenate(all_t)
    return total_loss / len(loader), compute_metrics(preds, tgts)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0; all_p = []; all_t = []
    for imgs, masks in _make_bar(loader, "val  "):
        imgs, masks = imgs.to(device), masks.to(device)
        out  = model(imgs)
        loss = criterion(out, masks)
        total_loss += loss.item()
        all_p.append(out.cpu().numpy())
        all_t.append(masks.cpu().numpy())
    preds = np.concatenate(all_p)
    tgts  = np.concatenate(all_t)
    return total_loss / len(loader), compute_metrics(preds, tgts)


In [8]:
# Main loop — Batch=4
SEED = SEED; FOLD = FOLD
print("\n" + "="*60)
print(f"  VARIANT : Batch=4")
print(f"  Seed={SEED} | Fold={FOLD} | BS={BATCH_SIZE} | LR={LR}")
print("="*60)
set_seed(SEED)
run_start = time.time()

# W&B init
wandb_run = wandb.init(
    project = "iconnet-ablation",
    name    = "Batch=4",
    config  = dict(
        variant    = "Batch=4",
        dataset    = DATASET_NAME,
        seed       = SEED,
        fold       = FOLD,
        img_size   = IMG_SIZE,
        batch_size = BATCH_SIZE,
        lr         = LR,
        epochs     = EPOCHS,
        patience   = PATIENCE,
    ),
    reinit = True,
)

full_train_ds = SegDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, IMG_SIZE, augment=False)
test_ds       = SegDataset(TEST_IMG_DIR,  TEST_MASK_DIR,  IMG_SIZE, augment=False)
dummy_labels  = np.arange(len(full_train_ds)) % 3

skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
splits = list(skf.split(np.zeros(len(full_train_ds)), dummy_labels))
tr_idx, vl_idx = splits[FOLD - 1]

train_ds = Subset(full_train_ds, tr_idx)
val_ds   = Subset(full_train_ds, vl_idx)
train_ds.dataset.augment = True

# num_workers=4 — same as Phase 2, one run per notebook so no conflict
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

print(f"  Train={len(tr_idx)} | Val={len(vl_idx)} | Test={len(test_ds)}")

model     = ICONNET(base=BASE_CH, dr=DROPOUT).to(device)
criterion = CombinedLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

best_val_dice = 0; patience_ctr = 0
ckpt_path = f"{SAVE_DIR}/best_Batcheq4.pth"
history   = {'train':[], 'val':[]}

print(f"  Checkpoint: {ckpt_path}")
print(f"  Training for up to {EPOCHS} epochs (patience={PATIENCE})")
print()

for epoch in range(EPOCHS):
    tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer, device)
    vl_loss, vl_m = eval_epoch(model, val_loader,   criterion, device)
    scheduler.step()
    history['train'].append({**tr_m,'loss':tr_loss})
    history['val'].append({**vl_m,'loss':vl_loss})

    # W&B per-epoch log
    wandb.log({
        'epoch'      : epoch + 1,
        'lr'         : optimizer.param_groups[0]['lr'],
        'train/loss' : tr_loss,
        **{f'train/{k}': v for k, v in tr_m.items()},
        'val/loss'   : vl_loss,
        **{f'val/{k}': v for k, v in vl_m.items()},
    }, step=epoch + 1)

    if (epoch+1) % 5 == 0:
        elapsed = time.time() - run_start
        print(f"  Ep{epoch+1:3d}/{EPOCHS} | "
              f"Tr Dice={tr_m['dice']:.4f} | "
              f"Vl Dice={vl_m['dice']:.4f} IoU={vl_m['iou']:.4f} | "
              f"Best={best_val_dice:.4f} Pat={patience_ctr}/{PATIENCE} | "
              f"Elapsed={str(timedelta(seconds=int(elapsed)))}", flush=True)

    if vl_m['dice'] > best_val_dice:
        best_val_dice = vl_m['dice']; patience_ctr = 0
        torch.save(model.state_dict(), ckpt_path)
        wandb.run.summary['best_val_dice'] = best_val_dice
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}", flush=True)
            break

run_elapsed = time.time() - run_start
print(f"\n  Run time: {str(timedelta(seconds=int(run_elapsed)))}")

# Load best checkpoint and evaluate
model.load_state_dict(torch.load(ckpt_path, map_location=device))
te_loss, te_m = eval_epoch(model, test_loader, criterion, device)

print(f"  Val  Dice : {best_val_dice:.4f}")
print(f"  Test Dice : {te_m['dice']:.4f}  IoU={te_m['iou']:.4f}  F1={te_m['f1']:.4f}", flush=True)

# W&B test metrics
wandb.log({**{f'test/{k}': v for k, v in te_m.items()}, 'test/loss': te_loss})
wandb.finish()
print("W&B run finished.")

result = {
    'variant'         : 'Batch=4',
    'val_dice'        : round(best_val_dice, 4),
    'test_dice'       : round(te_m['dice'], 4),
    'test_iou'        : round(te_m['iou'], 4),
    'test_f1'         : round(te_m['f1'], 4),
    'test_precision'  : round(te_m['precision'], 4),
    'test_recall'     : round(te_m['recall'], 4),
    'test_auc_roc'    : round(te_m['auc_roc'], 4),
    'test_specificity': round(te_m['specificity'], 4),
    'run_time_s'      : round(run_elapsed, 1),
}

# Save CSV
import shutil
csv_path = '/kaggle/working/abl_batch_4.csv'
pd.DataFrame([result]).to_csv(csv_path, index=False)
print(f"\nCSV saved -> abl_batch_4.csv")

# Print full table — same data as CSV
print()
print("="*110)
print("  ABLATION RESULT — Batch=4")
print("="*110)
cols  = ['variant','val_dice','test_dice','test_iou','test_f1','test_precision','test_recall','test_auc_roc','test_specificity','run_time_s']
hdr   = f"{'variant':<25} {'val_dice':>9} {'test_dice':>10} {'test_iou':>9} {'test_f1':>8} {'test_prec':>10} {'test_rec':>9} {'test_auc':>9} {'test_spec':>10} {'time_s':>7}"
print(hdr)
print("-"*110)
r = result
print(f"{str(r['variant']):<25} {r['val_dice']:>9.4f} {r['test_dice']:>10.4f} {r['test_iou']:>9.4f} {r['test_f1']:>8.4f} {r['test_precision']:>10.4f} {r['test_recall']:>9.4f} {r['test_auc_roc']:>9.4f} {r['test_specificity']:>10.4f} {r['run_time_s']:>7.1f}")
print("="*110)

# Verify downloadable
if os.path.exists(csv_path):
    size = os.path.getsize(csv_path)/1024
    print(f"\nREADY: abl_batch_4.csv ({size:.1f} KB) — Output tab -> download arrow")
else:
    print("WARNING: CSV not found!")



  VARIANT : Batch=4
  Seed=2024 | Fold=2 | BS=4 | LR=0.0001


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run indqsanf
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260503_202719-indqsanf
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Batch=4
wandb: ⭐️ View project at https://wandb.ai/ahmad-nafees-tihami-brac-university/iconnet-ablation
wandb: 🚀 View run at https://wandb.ai/ahmad-nafees-tihami-brac-university/iconnet-ablation/runs/indqsanf


  Train=3146 | Val=787 | Test=860
  Checkpoint: /kaggle/working/ablation_ckpts/best_Batcheq4.pth
  Training for up to 50 epochs (patience=7)



train:   0%|          | 0/787 [00:00<?, ?it/s]

val  :   0%|          | 0/197 [00:00<?, ?it/s]

...

  Early stop at epoch 37

  Run time: 6:21:17


val  :   0%|          | 0/215 [00:00<?, ?it/s]

  Val  Dice : 0.8473
  Test Dice : 0.8584  IoU=0.7519  F1=0.8584


wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:          epoch ▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
wandb:             lr █▇▇▆▄▃▂▂▁████▇▇▇▆▆▅▄▄▃▃▂▂▂▁▁▁███████▇
wandb:  test/accuracy ▁
wandb:    test/auc_pr ▁
wandb:   test/auc_roc ▁
wandb:      test/dice ▁
wandb:        test/f1 ▁
wandb:       test/iou ▁
wandb:      test/loss ▁
wandb: test/precision ▁
wandb:            +22 ...
wandb: 
wandb: Run summary:
wandb: best_val_dice 0.84727
wandb:         epoch 37
wandb:            lr 9e-05
wandb: test/accuracy 0.99456
wandb:   test/auc_pr 0.92913
wandb:  test/auc_roc 0.99594
wandb:     test/dice 0.85838
wandb:       test/f1 0.85838
wandb:      test/iou 0.7519
wandb:     test/loss 0.08934
wandb:           +23 ...
wandb: 
wandb: 🚀 View run Batch=4 at: https://wandb.ai/ahmad-nafees-tihami-brac-university/iconnet-ablation/runs/indqsanf
wandb: ⭐️ View project at: https://wandb.ai/ahmad-nafees-tihami-brac-university/iconnet-ablation
wandb: Synced 5 W&B file(s), 0 media file(s), 0 art

W&B run finished.

CSV saved -> abl_batch_4.csv

  ABLATION RESULT — Batch=4
variant                    val_dice  test_dice  test_iou  test_f1  test_prec  test_rec  test_auc  test_spec  time_s
--------------------------------------------------------------------------------------------------------------
Batch=4                      0.8473     0.8584    0.7519   0.8584     0.8509    0.8660    0.9959     0.9971 22877.5

READY: abl_batch_4.csv (0.2 KB) — Output tab -> download arrow
